# Cross-Source Consistency Check: Stage 1 (guided dataset) vs Stage 2 (Kaggle Transfermarkt data)

**Question this notebook answers.** The guided dataset (European Soccer database, up to 2016) and the Transfermarkt dataset (2012 to 2026) come from different sources with **different ID systems**. How do we know that appending them gives one consistent dataset?

**Answer in five checks**
1. **IDs are not comparable** (so the stages are never joined by ID): ranges differ and numbers can collide.
2. **No overlap and aligned seasons** at the Stage 1 / Stage 2 boundary, so no match is counted twice.
3. **Same real matches:** the Transfermarkt raw `games` table also covers 2012 to 2015, which overlaps the guided data. Matching on league + date + score shows both sources record the same matches.
4. **Team crosswalk:** from the matched fixtures we build a Stage 1 team ID to Transfermarkt club ID table, and check it is consistent and that names agree.
5. **Base rates are continuous** across the boundary, and the master dataset has unique, correctly prefixed match IDs.

**Inputs.** The two engineered stage files and the raw Transfermarkt `games.csv` (edit the paths below).

In [1]:
import re, difflib, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

STAGE1_FILE = Path("data/Stage1_PlayerAware_Football_2008_2016_Causal_v2.csv")
STAGE2_FILE = Path("data/Stage2_PlayerAware_Football_2016_2026_Causal_v2.csv")
GAMES_RAW   = Path("csv/stage_2/games.csv")          # raw Transfermarkt games table
OUT_DIR     = Path("cross_source_outputs"); OUT_DIR.mkdir(exist_ok=True)
PRIMARY = ["BE1", "ES1", "FR1", "GB1", "IT1", "L1", "NL1", "PO1", "SC1"]

s1 = pd.read_csv(STAGE1_FILE, parse_dates=["date"])
s2 = pd.read_csv(STAGE2_FILE, parse_dates=["date"])
print("Stage 1:", s1.shape, "| Stage 2:", s2.shape)

Stage 1: (25979, 62) | Stage 2: (29606, 62)


## 1. The ID systems are different (never join by ID)

In [2]:
ids1 = set(s1.home_team_id) | set(s1.away_team_id)
ids2 = set(s2.home_team_id) | set(s2.away_team_id)
print("match id ranges  : Stage 1 %d to %d | Stage 2 %d to %d" % (s1.game_id.min(), s1.game_id.max(), s2.game_id.min(), s2.game_id.max()))
print("match_uid prefix : Stage 1 ->", s1.match_uid.str[:3].unique().tolist(), "| Stage 2 ->", s2.match_uid.str[:3].unique().tolist())
print("team ids         : Stage 1 %d clubs | Stage 2 %d clubs | numbers used in BOTH systems: %d" % (len(ids1), len(ids2), len(ids1 & ids2)))

names1 = pd.concat([s1[["home_team_id", "home_team"]].set_axis(["id", "name"], axis=1),
                    s1[["away_team_id", "away_team"]].set_axis(["id", "name"], axis=1)]).drop_duplicates("id")
names2 = pd.concat([s2[["home_team_id", "home_team"]].set_axis(["id", "name"], axis=1),
                    s2[["away_team_id", "away_team"]].set_axis(["id", "name"], axis=1)]).drop_duplicates("id")
collisions = names1.merge(names2, on="id", suffixes=("_stage1", "_stage2"))
print("\nTeam ID numbers that mean DIFFERENT clubs in the two systems:")
display(collisions)
print("=> A join on team ID would silently attach the wrong club. The stages are harmonised on football concepts and appended by rows instead.")

match id ranges  : Stage 1 483129 to 2216672 | Stage 2 2697986 to 4856840
match_uid prefix : Stage 1 -> ['S1_'] | Stage 2 -> ['TM_']
team ids         : Stage 1 299 clubs | Stage 2 295 clubs | numbers used in BOTH systems: 1

Team ID numbers that mean DIFFERENT clubs in the two systems:


,id,name_stage1,name_stage2
0,8024,Widzew Łódź,FC Arouca


=> A join on team ID would silently attach the wrong club. The stages are harmonised on football concepts and appended by rows instead.


## 2. Boundary: no overlap, aligned season labels

In [3]:
print("Stage 1 dates:", s1.date.min().date(), "to", s1.date.max().date(), "| last season:", s1.season.iloc[s1.date.argmax()])
print("Stage 2 dates:", s2.date.min().date(), "to", s2.date.max().date(), "| first season:", s2.season.iloc[s2.date.argmin()])
print("Overlapping date range between the two output files:", s1.date.max() >= s2.date.min())
print("Season codes: Stage 1 season_start", s1.season_start.min(), "to", s1.season_start.max(), "| Stage 2", s2.season_start.min(), "to", s2.season_start.max())

master = pd.concat([s1, s2], ignore_index=True)
dup_uid = master.match_uid.duplicated().sum()
dup_fixture = master.duplicated(["league_code", "date", "home_team", "away_team"]).sum()
print("\nMaster rows:", len(master), "| duplicated match_uid:", dup_uid, "| duplicated (league, date, home, away):", dup_fixture)

Stage 1 dates: 2008-07-18 to 2016-05-25 | last season: 2015/2016
Stage 2 dates: 2016-07-29 to 2026-05-24 | first season: 2016/17
Overlapping date range between the two output files: False
Season codes: Stage 1 season_start 2008 to 2015 | Stage 2 2016 to 2025

Master rows: 55585 | duplicated match_uid: 0 | duplicated (league, date, home, away): 0


## 3. Do both sources record the same real matches? (2012 to 2015 overlap)
The raw Transfermarkt `games` table also contains the 2012 to 2015 league seasons, which the guided dataset covers too. For the nine primary leagues we compare **league + date + final score**, which needs no IDs or names.

In [4]:
g = pd.read_csv(GAMES_RAW, usecols=["game_id", "competition_id", "season", "date", "home_club_id", "away_club_id",
                                    "home_club_goals", "away_club_goals", "home_club_name", "away_club_name"])
g = g[g.competition_id.isin(PRIMARY) & g.season.between(2012, 2015) & g.home_club_goals.notna()].copy()
g["date"] = pd.to_datetime(g["date"])
a = s1[s1.league_code.isin(PRIMARY) & s1.season_start.between(2012, 2015)].copy()

ka = a.rename(columns={"audit_home_goals": "hg", "audit_away_goals": "ag", "league_code": "lc"})
kg = g.rename(columns={"competition_id": "lc", "home_club_goals": "hg", "away_club_goals": "ag"}).astype({"hg": int, "ag": int})
for d in (ka, kg):
    d["key"] = d.lc + "|" + d.date.astype(str) + "|" + d.hg.astype(str) + "|" + d.ag.astype(str)

counts = pd.concat([ka.key.value_counts(), kg.key.value_counts()], axis=1, keys=["stage1", "transfermarkt"]).fillna(0)
agree = int(counts.min(axis=1).sum())
print("Nine leagues, 2012-2015:")
print("  Stage 1 (guided) matches        :", len(ka))
print("  Transfermarkt matches           :", len(kg))
print("  identical league+date+score     :", agree)
print("  share of Stage 1 found in TM    : %.2f%%" % (100 * agree / len(ka)))
print("  share of TM found in Stage 1    : %.2f%%" % (100 * agree / len(kg)))

by_season = pd.DataFrame({"stage1": ka.groupby("season_start").size(), "transfermarkt": kg.groupby("season").size()})
by_season["difference"] = by_season.transfermarkt - by_season.stage1
display(by_season.T)
print("Differences are matches that exist in Transfermarkt but are missing from the guided database (for example gaps in 2013);")
print("no guided match disagrees with Transfermarkt on date or score except the small share above.")

Nine leagues, 2012-2015:
  Stage 1 (guided) matches        : 11263
  Transfermarkt matches           : 11492
  identical league+date+score     : 11244
  share of Stage 1 found in TM    : 99.83%
  share of TM found in Stage 1    : 97.84%


,2012,2013,2014,2015
stage1,2840,2612,2905,2906
transfermarkt,2840,2840,2906,2906
difference,0,228,1,0


Differences are matches that exist in Transfermarkt but are missing from the guided database (for example gaps in 2013);
no guided match disagrees with Transfermarkt on date or score except the small share above.


## 4. Team crosswalk (Stage 1 team ID to Transfermarkt club ID)
Built **only from the matched fixtures** (league + date + score, unique keys), never from names. Names are then used as an independent check.

In [5]:
def norm(x):
    x = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode().lower()
    x = re.sub(r"\b(fc|cf|ac|as|sc|sv|vfb|vfl|rc|ud|cd|1\.|fk|sk|kv|krc|ksc|rsc|sporting|club|de|the)\b", "", x)
    return re.sub(r"[^a-z0-9]", "", x)

ua = ka[~ka.duplicated("key", keep=False)]
ug = kg[~kg.duplicated("key", keep=False)]
m = ua.merge(ug, on="key", suffixes=("_s1", "_tm"))
print("fixtures with a unique league+date+score key in both sources:", len(m))

pairs = pd.concat([m[["home_team_id", "home_club_id"]].set_axis(["s1", "tm"], axis=1),
                   m[["away_team_id", "away_club_id"]].set_axis(["s1", "tm"], axis=1)])
votes = pairs.groupby(["s1", "tm"]).size().rename("supporting_fixtures").reset_index().sort_values("supporting_fixtures", ascending=False)
best = votes.drop_duplicates("s1")
n_conflict = int((votes.groupby("s1").size() > 1).sum())
n_tm_conflict = int((votes.groupby("tm").size() > 1).sum())
print("Stage 1 teams mapped:", best.s1.nunique(), "| Stage 1 teams with more than one Transfermarkt candidate:", n_conflict,
      "| Transfermarkt clubs claimed by more than one Stage 1 team:", n_tm_conflict)

tm_names = pd.concat([g[["home_club_id", "home_club_name"]].set_axis(["id", "name"], axis=1),
                      g[["away_club_id", "away_club_name"]].set_axis(["id", "name"], axis=1)]).drop_duplicates("id").set_index("id").name
s1_names = names1.set_index("id").name
crosswalk = best.rename(columns={"s1": "stage1_team_api_id", "tm": "transfermarkt_club_id"}).copy()
crosswalk["stage1_team_name"] = crosswalk.stage1_team_api_id.map(s1_names)
crosswalk["transfermarkt_club_name"] = crosswalk.transfermarkt_club_id.map(tm_names)
crosswalk["name_similarity"] = [round(difflib.SequenceMatcher(None, norm(x), norm(y)).ratio(), 3)
                                for x, y in zip(crosswalk.stage1_team_name, crosswalk.transfermarkt_club_name)]
crosswalk.to_csv(OUT_DIR / "team_id_crosswalk.csv", index=False)
print("Name agreement (independent check): median similarity %.2f | share with similarity >= 0.6: %.1f%%"
      % (crosswalk.name_similarity.median(), 100 * (crosswalk.name_similarity >= 0.6).mean()))
display(crosswalk.sort_values("name_similarity").head(8))
print("Lowest-similarity rows are name variants of the same club (e.g. different spellings or abbreviations); check them if you use the crosswalk.")

# clubs that continue across the boundary
tm_2016 = set(s2[s2.season_start == 2016].home_team_id) | set(s2[s2.season_start == 2016].away_team_id)
s1_2015 = set(s1[s1.season_start == 2015].home_team_id) | set(s1[s1.season_start == 2015].away_team_id)
mapped = crosswalk[crosswalk.stage1_team_api_id.isin(s1_2015)]
print("\nClubs in the guided data in 2015/16:", len(s1_2015), "| with a crosswalk entry:", len(mapped),
      "| of those playing again in Stage 2 (2016/17):", int(mapped.transfermarkt_club_id.isin(tm_2016).sum()))

fixtures with a unique league+date+score key in both sources: 9046
Stage 1 teams mapped: 212 | Stage 1 teams with more than one Transfermarkt candidate: 0 | Transfermarkt clubs claimed by more than one Stage 1 team: 0
Name agreement (independent check): median similarity 1.00 | share with similarity >= 0.6: 94.8%


,stage1_team_api_id,transfermarkt_club_id,supporting_fixtures,stage1_team_name,transfermarkt_club_name,name_similarity
105,8686,12,115,Roma,Associazione Sportiva Roma,0.286
71,8543,398,114,Lazio,Società Sportiva Lazio S.p.A.,0.357
193,10229,1090,121,AZ,AZ Alkmaar,0.364
95,8640,383,115,PSV,PSV Eindhoven,0.400
52,8464,467,90,N.E.C.,NEC Nijmegen,0.429
2,4087,14171,84,Évian Thonon Gaillard FC,Thonon Évian Grand Genève FC,0.439
84,8593,610,112,Ajax,Ajax Amsterdam,0.471
180,10167,130,80,Parma,Parma Calcio 1913,0.500


Lowest-similarity rows are name variants of the same club (e.g. different spellings or abbreviations); check them if you use the crosswalk.

Clubs in the guided data in 2015/16: 188 | with a crosswalk entry: 162 | of those playing again in Stage 2 (2016/17): 142


## 5. Base rates are continuous across the boundary

In [6]:
allm = pd.concat([s1.assign(stage="Stage 1"), s2.assign(stage="Stage 2")])
allm = allm[allm.league_code.isin(PRIMARY)]
rates = allm.groupby("season_start").agg(matches=("match_uid", "size"),
        goals_per_match=("audit_home_goals", lambda x: 0),
        ).drop(columns="goals_per_match")
rates["goals_per_match"] = allm.groupby("season_start").apply(lambda d: (d.audit_home_goals + d.audit_away_goals).mean(), include_groups=False)
for c in ["H", "D", "A"]:
    rates[f"{c}_%"] = allm.groupby("season_start").apply(lambda d: 100 * (d.target_result == c).mean(), include_groups=False)
display(rates.round(2).loc[2012:2019])
before = allm[allm.season_start.between(2012, 2015)]; after = allm[allm.season_start.between(2016, 2019)]
print("Home win %% 2012-2015: %.1f | 2016-2019: %.1f | goals per match: %.2f vs %.2f" % (
    100 * (before.target_result == "H").mean(), 100 * (after.target_result == "H").mean(),
    (before.audit_home_goals + before.audit_away_goals).mean(), (after.audit_home_goals + after.audit_away_goals).mean()))

,matches,goals_per_match,H_%,D_%,A_%
season_start,,,,,
2012,2840,2.81,44.47,26.23,29.30
2013,2612,2.77,46.40,23.81,29.79
2014,2905,2.67,44.89,25.34,29.78
2015,2906,2.74,44.32,25.26,30.42
2016,2906,2.78,47.38,23.88,28.73
2017,2876,2.76,45.34,24.17,30.49
2018,2876,2.82,45.65,24.10,30.25
2019,2674,2.78,44.80,24.12,31.08


Home win % 2012-2015: 45.0 | 2016-2019: 45.8 | goals per match: 2.75 vs 2.78


## Summary
* The two sources use **different, non-comparable ID systems** (and at least one team ID number is shared by two different clubs), so the datasets are **never merged on IDs**. They are harmonised on football concepts and appended by rows, with `S1_` and `TM_` prefixes keeping every match unique.
* The sources **describe the same real matches**: in the 2012 to 2015 overlap, almost every guided match has an identical league, date and score in Transfermarkt.
* A **team crosswalk** derived from those matches is internally consistent and agrees with club names.
* The boundary at mid-2016 has **no overlap and no duplicates**, and outcome and goal rates are continuous across it.
* The remaining difference between the stages is the **player-quality measure** (FIFA rating vs market value), which is handled by the common 0-100 percentile scale, the `data_source` column (kept for analysis, not used as a model input) and the modern-only robustness check.